# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")


## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# List all record sets via their @id
print("Available Record Sets (by @id):\n-----------------------------")
record_sets = []
for record_set in dataset.record_sets:
    print(f"@id: {record_set['@id']}, name: {record_set.get('name', 'N/A')}")
    record_sets.append(record_set['@id'])

if not record_sets:
    print("No record sets found. Fetching file objects...")
    # Try to list file objects as potential sources
    for file_obj in dataset.files:
        print(f"File @id: {file_obj['@id']}, name: {file_obj.get('name', 'N/A')}, encoding: {file_obj.get('encodingFormat', 'N/A')}")
    # For demonstration, take the first tabular file as the main record set
    first_file = next((f for f in dataset.files if f.get('encodingFormat', '').startswith('text/csv')), None)
    if first_file:
        record_sets = [first_file['@id']]
        print(f"\nUsing FileObject @id as record_set: {first_file['@id']}")

# For each record set, print list of fields
if record_sets:
    print("\nFields for each record set:")
    for rs_id in record_sets:
        print(f"--- Record Set {rs_id} ---")
        fields = dataset.fields(record_set=rs_id)
        for field in fields:
            print(f"  Field @id: {field['@id']}  -  {field.get('name', 'N/A')}  (dataType: {field.get('dataType', 'N/A')})")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Assign the record_set to use (from previous code - update if more record_sets exist)
record_sets = record_sets  # From above
dataframes = {}

# Load records from each record set
for record_set in record_sets:
    try:
        records_iter = dataset.records(record_set=record_set)
        records = list(records_iter)
        if records:
            dataframes[record_set] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for {record_set}")
        else:
            print(f"No records found in {record_set}.\n")
    except Exception as e:
        print(f"Could not load records from {record_set}: {e}")

# Display columns for the first DataFrame
main_record_set = list(dataframes.keys())[0]
print(f"Columns in DataFrame for record set {main_record_set}:")
print(dataframes[main_record_set].columns.tolist())

# Show data head
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field for analysis (update if needed)
df = dataframes[main_record_set]

# Try to automatically find a candidate numeric field (e.g. Molecular Weight, Coverage, Peptide Count, etc.)
possible_numeric_ids = [col for col in df.columns if any(x in col.lower() for x in ['weight', 'mw', 'count', 'coverage', 'amount', 'abundance', 'intensity', 'peptide'])]
print(f"Candidate numeric columns: {possible_numeric_ids}")

if possible_numeric_ids:
    numeric_field = possible_numeric_ids[0]  # For example, choose the first
else:
    numeric_field = df.select_dtypes('number').columns[0]

# Inspect value distribution
print(f"Value statistics for field {numeric_field}:")
print(df[numeric_field].describe())

# Remove NaNs for filtering
dn = df[[numeric_field]].dropna()
threshold = dn[numeric_field].median()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3g} (median):")
print(filtered_df[[numeric_field]].head())

# Add normalized column
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a categorical/grouping field
possible_group_fields = [col for col in df.columns if any(x in col.lower() for x in ['type', 'sample', 'description', 'protein', 'accession', 'category', 'group'])]
print(f"Candidate group fields: {possible_group_fields}")

if possible_group_fields:
    group_field = possible_group_fields[0]
    try:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    except Exception as e:
        print(f"Could not group by {group_field}: {e}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if possible_group_fields:
    group_field = possible_group_fields[0]
    plt.figure(figsize=(12,4))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 mass spectrometry dataset was loaded via its Croissant schema (@id references used throughout).
- We successfully extracted tabular data, identified key numeric fields, and performed filtering and normalization.
- Distribution and grouping visualizations revealed trends and potential grouping effects in protein-related measurements.
- This notebook can be extended for further domain-specific analysis, including protein clustering, peptide spectrum matching, or cross-sample comparisons.
